# Building Initial Board Setup

In [1]:
from IPython.display import clear_output
import time
import init
import excavate
import explore
import subactions

game_state, player1_hand, player1_mat = init.initialize_game()

Your dragons cards: [146, 180, 9]
Your cave cards: [38, 23, 25]

Randomly assigned starting resources: {'milk': 0, 'crystal': 3, 'gold': 0, 'meat': 0}
Your dragon cards: [146, 180]
Your cave cards: [23, 25]


In [4]:
game_state.showcase

{'dragons': [19, 145, 85], 'caves': [8, 24, 66]}

## Game state reference

`initialize_game()` returns a shared `GameState` plus player-specific state:

- **`game_state`**: A dataclass containing board-level state shared by all players:
  - `game_state.dragons`: Dragon card database as a pandas DataFrame.
  - `game_state.caves`: Cave card database as a pandas DataFrame.
  - `game_state.showcase`: Face-up dragon and cave card IDs available to players.
  - `game_state.guildtrack`: The 12 guild-track rewards.
  - `game_state.dragon_deck_ids`: Dragon IDs remaining after showcase and starting-hand cards are removed.
  - `game_state.cave_deck_ids`: Cave IDs remaining after showcase and starting-hand cards are removed.

For example, inspect the dragon showcase with `game_state.showcase["dragons"]`. These board-level values are accessed and modified through `game_state`, not returned as separate values.

- **`player1_hand`**: A dictionary containing the player's name, coins, cards in hand, eggs, resources, and guild position.
- **`player1_mat`**: The player's personal board for placing dragons and excavating caves. It contains a 3-by-4 `dragons` grid and a 3-by-3 `excavated` grid.

Call `init.initialize_game()` again to create a new independent `GameState` for another game.

In [3]:
player1_hand["coins"] = 6
def pass_turns():
    print("All future turns are passed for this round")
    return

### Start first round

In [ ]:
actions = { "0": pass_turns,
            "1": excavate.Excavate, 
            #"2": Entice, 
            #"3": Explore, 
            }

while player1_hand["coins"] > 0:
    player1_hand["coins"] -= 1  # Spend a coin for the turn
    clear_output(wait=True)
    time.sleep(0.1)  # Small delay to ensure clear_output completes
    
    print("Choose an action:\n1. Excavate \n2. Entice a dragon\n3. Explore a cave")
    
    # Validate user input - must be 1, 2, or 3
    valid_choice = False
    while not valid_choice:
        action = input("Enter the number of your chosen action (1-3): ")
        if action in ["1", "2", "3"]:
            valid_choice = True
        elif action == "0":
            break
        else:
            print("Invalid choice! Please enter 1, 2, or 3")
    # Depending on action choosen, call function
    actions[action](game_state, player1_hand, player1_mat)

    # Do end of turn checks


    print(f"You chose action {action}")
    input("Press Enter to continue...")

Choose an action:
1. Excavate 
2. Entice a dragon
3. Explore a cave
Invalid choice! Please enter 1, 2, or 3


KeyError: '2'

In [3]:
player1_hand["hand_caves"]

[3, 52]

## Plan: Centralize Shared Game State

Keep the showcase and decks as board-level state, but do not expose them as independent module globals. Introduce one shared game-state container initialized by `init.initialize_game()` and make board-mutating actions receive that state explicitly. This keeps the state accessible to any action without coupling actions to import-time globals, and it supports restarting or running multiple games.

**Steps**

5. Add small access/mutation helpers if repeated operations emerge, such as draw-from-deck, take-from-showcase, and replace-showcase-card. These helpers should enforce removal from the deck/showcase consistently rather than allowing arbitrary list mutation throughout actions.
6. Update the initialization/state-reference documentation in `game_engine.ipynb` or `README.md` to show how to inspect and modify `game_state.showcase`, `game_state.dragon_deck_ids`, and `game_state.cave_deck_ids`, and document that a new game gets a new state object.

**Relevant files**
- `/home/ozzy/Documents/prog/Wyrmspan-Bot/init.py` — owns initialization; add the shared state type/container and construct it in `initialize_game()`.
- `/home/ozzy/Documents/prog/Wyrmspan-Bot/subactions.py` — owns card-related subactions; replace the implicit `global guildtrack` approach and thread state into deck/showcase actions.
- `/home/ozzy/Documents/prog/Wyrmspan-Bot/excavate.py` — forwards state into cave-card abilities and any future cave/dragon draw operations.
- `/home/ozzy/Documents/prog/Wyrmspan-Bot/explore.py` — forwards state through benefit activation when a benefit needs shared board state.
- `/home/ozzy/Documents/prog/Wyrmspan-Bot/game_engine.ipynb` — current initialization and state documentation/call site.
- `/home/ozzy/Documents/prog/Wyrmspan-Bot/README.md` — update usage guidance if it documents initialization.

**Verification**
1. Initialize a game and assert the state contains exactly three dragon and three cave showcase IDs, with those IDs absent from their respective decks.
2. Initialize a second game and verify its state is independent: changing the first game's showcase/deck does not change the second game's values.
3. Exercise one action that draws or takes a card and verify both the source collection and destination collection update together.
4. Run a syntax/import check for `init.py`, `subactions.py`, `excavate.py`, and `explore.py`, then execute the relevant notebook or a small Python smoke test.
5. Search for remaining `global guildtrack`, direct deck/showcase globals, and stale tuple unpacking after migration.

**Decisions**
- Recommended: one explicit shared state object, not several module-level globals.
- Showcase and decks are global to a game/board, not global to the Python process.
- Player hands and mats remain explicit player state; do not globalize them.
- A short compatibility phase for the existing tuple return is acceptable, but the final API should avoid eight unrelated return values.

**Further Considerations**
1. If only one game will ever exist and this is a quick prototype, module globals can work, but they require `global` declarations for rebinding and make tests/restarts brittle. The state object is still the safer small change.
2. If multiplayer is the next feature, use a `GameState` with a `players` collection now; otherwise keep the first refactor focused on board state and avoid redesigning player initialization prematurely.


## Old Notes

## Game state reference

`initialize_game()` returns the following pieces of game state:

- **`dragons`**: A pandas DataFrame containing the dragon card database. Each row represents a dragon, including its `id`, name, costs, points, habitats, and abilities.
- **`caves`**: A pandas DataFrame containing the cave card database. Each row represents a cave, including its `id`, costs, points, and card abilities.
- **`showcase`**: A dictionary showing the face-up cards currently available to players. It has two keys:
  - `dragons`: a list of dragon card IDs in the showcase.
  - `caves`: a list of cave card IDs in the showcase.
- **`guildtrack`**: A list of 12 guild-track spaces. Each element describes the reward or action triggered when a player reaches that space, such as gaining a VP, egg, resource, dragon card, cave card, or coin.
- **`player1_hand`**: A dictionary containing the player's cards and resources:
  - `name`: the player's name.
  - `coins`: coins available to spend on turns and other effects. The starting value is 6.
  - `hand_dragons`: dragon card IDs held in hand. The starting hand contains 2 dragons after discarding 1 of the 3 drawn cards.
  - `hand_caves`: cave card IDs held in hand. The starting hand contains 2 caves after discarding 1 of the 3 drawn cards.
  - `eggs`: eggs available to pay excavation costs or use for other effects. The starting value is 2.
  - `resources`: a dictionary containing counts of the four general resources: `milk`, `crystal`, `gold`, and `meat`. The starting total is 3 resources.
  - `guild_position`: the player's current position on the guild track. The starting position is 0.
- **`player1_mat`**: The player's personal 3-by-4 board for placing dragons and excavating caves. It contains:
  - `dragons`: a 3-by-4 grid of dragon IDs, with `0` for empty spaces.
  - `excavated`: a 3-by-3 grid for cave excavations in columns 2-4, with `0` for spaces that have not been excavated and a cave ID for excavated spaces.
- **`dragon_deck_ids`**: A list of dragon card IDs still in the dragon deck after showcase and starting-hand cards have been removed.
- **`cave_deck_ids`**: A list of cave card IDs still in the cave deck after showcase and starting-hand cards have been removed.
